In [1]:
import numpy as np
import tensorflow as tf
from ogb.nodeproppred import Evaluator, NodePropPredDataset
from tensorflow.keras.layers import BatchNormalization, Dropout, Input
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from spektral.datasets.ogb import OGB
from spektral.layers import GCNConv
from spektral.transforms import AdjToSpTensor, GCNFilter

In [3]:
dataset_name = "ogbn-arxiv"
ogb_dataset = NodePropPredDataset(dataset_name)
dataset = OGB(ogb_dataset, transforms=[GCNFilter(), AdjToSpTensor()])
graph = dataset[0]
x, adj, y = graph.x, graph.a, graph.y

Using exist file arxiv.zip
Extracting dataset\arxiv.zip


BadZipFile: File is not a zip file

In [3]:
data = TUDataset("PROTEINS")

100%|█████████████████████████████████████████| 447k/447k [00:01<00:00, 368kB/s]
D:\ProgramData\Anaconda3\envs\auto2\lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Successfully loaded PROTEINS.


D:\ProgramData\Anaconda3\envs\auto2\lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [4]:
np.random.shuffle(data)
split = int(0.8 * len(data))
data_tr, data_te = data[:split], data[split:]

# Data loaders
loader_tr = DisjointLoader(data_tr, batch_size=batch_size, epochs=epochs)
loader_te = DisjointLoader(data_te, batch_size=batch_size)

C:\Users\dell\AppData\Local\Temp\ipykernel_5984\1351671128.py:1: UserWarning: you are shuffling a 'TUDataset' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  np.random.shuffle(data)


In [8]:
model = GeneralGNN(data.n_labels, activation="softmax")
optimizer = Adam(learning_rate)
loss_fn = CategoricalCrossentropy()

In [9]:
def train_step(inputs, target):
    with tf.GradientTape() as tape:
        predictions = model(inputs, training=True)
        loss = loss_fn(target, predictions) + sum(model.losses)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    acc = tf.reduce_mean(categorical_accuracy(target, predictions))
    return loss, acc

In [10]:
def evaluate(loader):
    output = []
    step = 0
    while step < loader.steps_per_epoch:
        step += 1
        inputs, target = loader.__next__()
        pred = model(inputs, training=False)
        outs = (
            loss_fn(target, pred),
            tf.reduce_mean(categorical_accuracy(target, pred)),
            len(target),  # Keep track of batch size
        )
        output.append(outs)
        if step == loader.steps_per_epoch:
            output = np.array(output)
            return np.average(output[:, :-1], 0, weights=output[:, -1])

In [11]:
epoch = step = 0
results = []
for batch in loader_tr:
    step += 1
    loss, acc = train_step(*batch)
    results.append((loss, acc))
    if step == loader_tr.steps_per_epoch:
        step = 0
        epoch += 1
        results_te = evaluate(loader_te)
        print(
            "Ep. {} - Loss: {:.3f} - Acc: {:.3f} - Test loss: {:.3f} - Test acc: {:.3f}".format(
                epoch, *np.mean(results, 0), *results_te
            )
        )
        results = []

D:\ProgramData\Anaconda3\envs\auto2\lib\site-packages\spektral\data\utils.py:221: UserWarning: you are shuffling a 'TUDataset' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  np.random.shuffle(a)


Ep. 1 - Loss: 0.627 - Acc: 0.648 - Test loss: 96.889 - Test acc: 0.426
Ep. 2 - Loss: 0.548 - Acc: 0.712 - Test loss: 7.128 - Test acc: 0.439
Ep. 3 - Loss: 0.521 - Acc: 0.751 - Test loss: 2.912 - Test acc: 0.453
Ep. 4 - Loss: 0.512 - Acc: 0.753 - Test loss: 0.701 - Test acc: 0.574
Ep. 5 - Loss: 0.520 - Acc: 0.754 - Test loss: 0.545 - Test acc: 0.722
Ep. 6 - Loss: 0.508 - Acc: 0.754 - Test loss: 0.573 - Test acc: 0.753
Ep. 7 - Loss: 0.501 - Acc: 0.756 - Test loss: 0.573 - Test acc: 0.731
Ep. 8 - Loss: 0.497 - Acc: 0.758 - Test loss: 0.544 - Test acc: 0.713
Ep. 9 - Loss: 0.488 - Acc: 0.760 - Test loss: 0.564 - Test acc: 0.691
Ep. 10 - Loss: 0.501 - Acc: 0.764 - Test loss: 0.568 - Test acc: 0.695
Ep. 11 - Loss: 0.491 - Acc: 0.770 - Test loss: 0.613 - Test acc: 0.641
Ep. 12 - Loss: 0.508 - Acc: 0.763 - Test loss: 0.564 - Test acc: 0.744
Ep. 13 - Loss: 0.486 - Acc: 0.782 - Test loss: 0.555 - Test acc: 0.704
Ep. 14 - Loss: 0.461 - Acc: 0.798 - Test loss: 0.569 - Test acc: 0.735
Ep. 15 - Loss:

Ep. 117 - Loss: 0.097 - Acc: 0.971 - Test loss: 0.857 - Test acc: 0.619
Ep. 118 - Loss: 0.118 - Acc: 0.953 - Test loss: 0.934 - Test acc: 0.623
Ep. 119 - Loss: 0.121 - Acc: 0.959 - Test loss: 0.972 - Test acc: 0.673
Ep. 120 - Loss: 0.104 - Acc: 0.968 - Test loss: 0.760 - Test acc: 0.749
Ep. 121 - Loss: 0.139 - Acc: 0.945 - Test loss: 0.813 - Test acc: 0.722
Ep. 122 - Loss: 0.110 - Acc: 0.958 - Test loss: 1.220 - Test acc: 0.623
Ep. 123 - Loss: 0.127 - Acc: 0.956 - Test loss: 2.407 - Test acc: 0.453
Ep. 124 - Loss: 0.100 - Acc: 0.961 - Test loss: 4.082 - Test acc: 0.426
Ep. 125 - Loss: 0.118 - Acc: 0.954 - Test loss: 1.021 - Test acc: 0.650
Ep. 126 - Loss: 0.089 - Acc: 0.965 - Test loss: 1.365 - Test acc: 0.664
Ep. 127 - Loss: 0.130 - Acc: 0.950 - Test loss: 1.193 - Test acc: 0.677
Ep. 128 - Loss: 0.108 - Acc: 0.962 - Test loss: 1.163 - Test acc: 0.641
Ep. 129 - Loss: 0.103 - Acc: 0.957 - Test loss: 0.989 - Test acc: 0.682
Ep. 130 - Loss: 0.085 - Acc: 0.972 - Test loss: 1.320 - Test acc

Ep. 231 - Loss: 0.067 - Acc: 0.984 - Test loss: 1.328 - Test acc: 0.664
Ep. 232 - Loss: 0.114 - Acc: 0.963 - Test loss: 4.188 - Test acc: 0.444
Ep. 233 - Loss: 0.096 - Acc: 0.964 - Test loss: 1.304 - Test acc: 0.659
Ep. 234 - Loss: 0.086 - Acc: 0.976 - Test loss: 1.348 - Test acc: 0.709
Ep. 235 - Loss: 0.053 - Acc: 0.982 - Test loss: 1.029 - Test acc: 0.659
Ep. 236 - Loss: 0.047 - Acc: 0.981 - Test loss: 1.264 - Test acc: 0.668
Ep. 237 - Loss: 0.054 - Acc: 0.984 - Test loss: 2.467 - Test acc: 0.511
Ep. 238 - Loss: 0.065 - Acc: 0.980 - Test loss: 1.115 - Test acc: 0.650
Ep. 239 - Loss: 0.046 - Acc: 0.984 - Test loss: 1.223 - Test acc: 0.691
Ep. 240 - Loss: 0.050 - Acc: 0.989 - Test loss: 1.610 - Test acc: 0.632
Ep. 241 - Loss: 0.068 - Acc: 0.975 - Test loss: 1.633 - Test acc: 0.623
Ep. 242 - Loss: 0.026 - Acc: 0.993 - Test loss: 1.528 - Test acc: 0.619
Ep. 243 - Loss: 0.054 - Acc: 0.983 - Test loss: 1.648 - Test acc: 0.605
Ep. 244 - Loss: 0.058 - Acc: 0.979 - Test loss: 2.149 - Test acc

Ep. 345 - Loss: 0.038 - Acc: 0.989 - Test loss: 1.942 - Test acc: 0.664
Ep. 346 - Loss: 0.039 - Acc: 0.986 - Test loss: 1.512 - Test acc: 0.668
Ep. 347 - Loss: 0.014 - Acc: 0.994 - Test loss: 1.653 - Test acc: 0.664
Ep. 348 - Loss: 0.047 - Acc: 0.985 - Test loss: 1.503 - Test acc: 0.695
Ep. 349 - Loss: 0.028 - Acc: 0.990 - Test loss: 1.365 - Test acc: 0.704
Ep. 350 - Loss: 0.020 - Acc: 0.994 - Test loss: 1.476 - Test acc: 0.650
Ep. 351 - Loss: 0.043 - Acc: 0.984 - Test loss: 1.680 - Test acc: 0.682
Ep. 352 - Loss: 0.016 - Acc: 0.996 - Test loss: 1.389 - Test acc: 0.709
Ep. 353 - Loss: 0.038 - Acc: 0.988 - Test loss: 1.358 - Test acc: 0.704
Ep. 354 - Loss: 0.030 - Acc: 0.988 - Test loss: 0.915 - Test acc: 0.740
Ep. 355 - Loss: 0.028 - Acc: 0.990 - Test loss: 1.191 - Test acc: 0.655
Ep. 356 - Loss: 0.017 - Acc: 0.996 - Test loss: 1.448 - Test acc: 0.632
Ep. 357 - Loss: 0.018 - Acc: 0.993 - Test loss: 1.625 - Test acc: 0.628
Ep. 358 - Loss: 0.009 - Acc: 0.998 - Test loss: 1.550 - Test acc